In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/btc_features.csv", parse_dates=["time"])

# Target: did the NEXT hour close up? (1 = yes, 0 = no)
df["next_ret"] = df["return"].shift(-1)
df["next_up"] = (df["next_ret"] > 0).astype(int)
df = df.iloc[:-1]   # last row has no next hour yet

# Time-based split: first 80% to train, last 20% to test
split = int(len(df) * 0.8)
train = df.iloc[:split]
test = df.iloc[split:]

print("Train:", train.shape, train["time"].min(), "->", train["time"].max())
print("Test: ", test.shape, test["time"].min(), "->", test["time"].max())

Train: (3438, 10) 2026-05-09 14:00:00 -> 2026-09-29 19:00:00
Test:  (860, 10) 2026-09-29 20:00:00 -> 2026-11-04 15:00:00


In [4]:
print("Share of up hours in train:", round(train["next_up"].mean() * 100, 1), "%")
print("Share of up hours in test: ", round(test["next_up"].mean() * 100, 1), "%")

Share of up hours in train: 49.9 %
Share of up hours in test:  49.3 %


In [5]:
def accuracy(pred, actual):
    return round((pred == actual).mean() * 100, 1)

y_test = test["next_up"]

# Baseline 1: always predict whichever class was more common in TRAIN
majority = int(train["next_up"].mean() > 0.5)
pred1 = pd.Series(majority, index=test.index)
print("Always", "up" if majority else "down", ":", accuracy(pred1, y_test), "%")

# Baseline 2: next hour goes the same direction as this hour
pred2 = (test["return"] > 0).astype(int)
print("Same as last hour:    ", accuracy(pred2, y_test), "%")

# Baseline 3: next hour goes the opposite direction
pred3 = 1 - pred2
print("Opposite of last hour:", accuracy(pred3, y_test), "%")

Always down : 50.7 %
Same as last hour:     48.6 %
Opposite of last hour: 51.4 %


In [6]:
for name, part in [("train", train), ("test", test)]:
    same = ((part["return"] > 0).astype(int) == part["next_up"]).mean()
    print(name, "| same-as-last accuracy:", round(same * 100, 1), "%",
          "| opposite:", round((1 - same) * 100, 1), "%")

train | same-as-last accuracy: 47.3 % | opposite: 52.7 %
test | same-as-last accuracy: 48.6 % | opposite: 51.4 %


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

feature_cols = ["return", "ret_5", "vol_20", "rsi_14", "vol_ratio", "dist_ma"]

X_train, y_train = train[feature_cols], train["next_up"]
X_test, y_test = test[feature_cols], test["next_up"]

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_train, y_train)

pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

print("Train accuracy:", round((pred_train == y_train).mean() * 100, 1), "%")
print("Test accuracy: ", round((pred_test == y_test).mean() * 100, 1), "%")
print("Share of 'up' predictions in test:", round(pred_test.mean() * 100, 1), "%")

Train accuracy: 52.3 %
Test accuracy:  51.0 %
Share of 'up' predictions in test: 41.7 %


In [9]:
coefs = pd.Series(model[-1].coef_[0], index=feature_cols).round(3)
print(coefs.sort_values())

ret_5       -0.114
rsi_14      -0.070
return      -0.068
vol_20      -0.010
vol_ratio    0.053
dist_ma      0.090
dtype: float64


In [10]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
xgb.fit(X_train, y_train)

pred_train_x = xgb.predict(X_train)
pred_test_x = xgb.predict(X_test)

print("Train accuracy:", accuracy(pred_train_x, y_train), "%")
print("Test accuracy: ", accuracy(pred_test_x, y_test), "%")
print("Share of 'up' predictions in test:", round(pred_test_x.mean() * 100, 1), "%")

Train accuracy: 69.1 %
Test accuracy:  51.0 %
Share of 'up' predictions in test: 51.7 %


In [11]:
imp = pd.Series(xgb.feature_importances_, index=feature_cols).round(3)
print(imp.sort_values(ascending=False))

return       0.176
vol_ratio    0.175
ret_5        0.170
vol_20       0.161
dist_ma      0.160
rsi_14       0.158
dtype: float32


In [12]:
results = pd.DataFrame({
    "model": ["Always down", "Same as last hour", "Opposite of last hour",
              "Logistic Regression", "XGBoost"],
    "test_accuracy_%": [
        accuracy(pred1, y_test), accuracy(pred2, y_test), accuracy(pred3, y_test),
        accuracy(pred_test, y_test), accuracy(pred_test_x, y_test),
    ],
})
print(results.to_string(index=False))

                model  test_accuracy_%
          Always down             50.7
    Same as last hour             48.6
Opposite of last hour             51.4
  Logistic Regression             51.0
              XGBoost             51.0
